### RAG pipeline- data ingestion to vector D pipeline

In [6]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
### Read all the pdf inside the directory 
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: PaPer-Final.pdf
  ✓ Loaded 5 pages

Processing: R2OM66A01_Assignment - Design Problem(s)_MKT905_MKT 905.pdf
  ✓ Loaded 7 pages

Processing: Week-10.pdf
  ✓ Loaded 98 pages

Total documents loaded: 110


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-30T17:00:08+00:00', 'author': '', 'keywords': '', 'moddate': '2025-11-30T17:00:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\PaPer-Final.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'PaPer-Final.pdf', 'file_type': 'pdf'}, page_content='BiLSTM-based Language Identification in\nCode-Switched Social Media Text\nShouiya Tayal\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nshouriyatayal1234@gmail.com\nShivam Garg\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\ngarg05shivam@gmail.com\nN N Ajay Kumar\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nnagaajaykumar19@gmail.com\nEn

In [14]:
### TEXT splitting gets into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """split documents into smaller chunks for better RaG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of chunk
    print("\nExample chunk:")
    print(f"content: {split_docs[0].page_content[:200]}...")  # Show first 200 characters
    print(f"metadata: {split_docs[0].metadata}")
    return split_docs

In [16]:
chunks=split_documents(all_pdf_documents)
chunks

Split 110 documents into 142 chunks

Example chunk:
content: BiLSTM-based Language Identification in
Code-Switched Social Media Text
Shouiya Tayal
Computer Science and Engineering Department
Lovely Professional University
Phagwara, India
shouriyatayal1234@gmail...
metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-30T17:00:08+00:00', 'author': '', 'keywords': '', 'moddate': '2025-11-30T17:00:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\PaPer-Final.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'PaPer-Final.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-30T17:00:08+00:00', 'author': '', 'keywords': '', 'moddate': '2025-11-30T17:00:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\PaPer-Final.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'PaPer-Final.pdf', 'file_type': 'pdf'}, page_content='BiLSTM-based Language Identification in\nCode-Switched Social Media Text\nShouiya Tayal\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nshouriyatayal1234@gmail.com\nShivam Garg\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\ngarg05shivam@gmail.com\nN N Ajay Kumar\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nnagaajaykumar19@gmail.com\nEn

### Embedding and vectorstoreDB

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [26]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5636.26it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\shiva\AppData\Local\Temp\ipykernel_9708\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vectore store

In [27]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [28]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-30T17:00:08+00:00', 'author': '', 'keywords': '', 'moddate': '2025-11-30T17:00:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\PaPer-Final.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'PaPer-Final.pdf', 'file_type': 'pdf'}, page_content='BiLSTM-based Language Identification in\nCode-Switched Social Media Text\nShouiya Tayal\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nshouriyatayal1234@gmail.com\nShivam Garg\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\ngarg05shivam@gmail.com\nN N Ajay Kumar\nComputer Science and Engineering Department\nLovely Professional University\nPhagwara, India\nnagaajaykumar19@gmail.com\nEn

In [29]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 142 texts...


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.30it/s]


Generated embeddings with shape: (142, 384)
Adding 142 documents to vector store...
Successfully added 142 documents to vector store
Total documents in collection: 142
